<a href="https://colab.research.google.com/github/Mahalakshmi2008/ResearchMate-AI/blob/main/agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



<a href="https://colab.research.google.com/github/aditi-govindu/gemini-recipe-agent/blob/main/agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

Paste your Gemini API key and press Enter: ··········
API key set.


In [4]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [6]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
DOCUMENTS = {
    "sample_protocol": (
        "Students must fill in documents in this section."
        "Documents can be related to any topic - sports, manga, food, music etc."
        "Have a minimum of 4 sentences with each document."
        "Duplicate this for a minimum of 4 documents to test your agent."
    ),

    "fill_in1": (
        "Masala Dosa is a popular South Indian breakfast made with dosa batter and a spicy potato filling. "
        "To prepare the filling, cook potatoes and mix them with onions, green chillies, mustard seeds, curry leaves, and spices. "
        "Spread the dosa batter thinly on a hot pan and cook until it becomes crispy. "
        "Place the potato filling inside the dosa, fold it, and serve it hot with coconut chutney and sambar."
    ),

    "fill_in2": (
        "Vegetable Biryani is a flavorful rice dish prepared with basmati rice and mixed vegetables. "
        "First, cook onions and whole spices such as cinnamon, cloves, and cardamom in oil. "
        "Add vegetables, spices, and partially cooked basmati rice, then cook everything together until the rice is completely done. "
        "Vegetable biryani can be served hot with raita or a vegetable side dish."
    ),

    "fill_in3": (
        "Paneer Butter Masala is a creamy Indian dish made with paneer and a tomato-based gravy. "
        "Prepare the gravy using tomatoes, onions, ginger, garlic, butter, and Indian spices. "
        "Add paneer cubes and a little cream to the cooked gravy and simmer for a few minutes. "
        "Serve paneer butter masala hot with naan, roti, chapati, or rice."
    ),

    "fill_in4": (
        "Lemon Rice is a simple South Indian rice dish with a fresh lemon flavor. "
        "Cooked rice is mixed with a tempering made from mustard seeds, urad dal, chana dal, curry leaves, green chillies, and peanuts. "
        "Add turmeric and fresh lemon juice after turning off the heat and mix everything well. "
        "Lemon rice can be served as a quick breakfast, lunch, or snack."
    )
}

In [7]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check
print(search_documents("minimum of 4"))

[Source: sample_protocol]
Students must fill in documents in this section.Documents can be related to any topic - sports, manga, food, music etc.Have a minimum of 4 sentences with each document.Duplicate this for a minimum of 4 documents to test your agent.

[Source: fill_in4]
Lemon Rice is a simple South Indian rice dish with a fresh lemon flavor. Cooked rice is mixed with a tempering made from mustard seeds, urad dal, chana dal, curry leaves, green chillies, and peanuts. Add turmeric and fresh lemon juice after turning off the heat and mix everything well. Lemon rice can be served as a quick breakfast, lunch, or snack.


In [29]:

# @title Ask Gemini to search for something

import google.generativeai as genai
import os
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    model_name="gemini-3.8-flash",
    tools=[search_documents],
    system_instruction=(
        """
        You are a chef training at a Michelin star restaurant.
        When you need specific information about temperatures,
        ingredients to use, cook time etc. use the search_documents tool.
        Only answer based on what you find in the documents.
        If the documents do not contain the answer, say so clearly.
        """
    )
)

print("Model ready:", model.model_name)


Model ready: models/gemini-3.8-flash


In [30]:
import time
import google.generativeai as genai
import os

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

def run_agent(prompt: str):
    print(f"User Prompt: {prompt}")

    # Use the configured GenerativeModel with search_documents tool
    chat = model.start_chat(enable_automatic_function_calling=True)
    response = chat.send_message(prompt)

    # Display the final response
    print(f"Agent Response:\n{response.text}")

In [ ]:
# @title Ask follow up questions
import google.api_core.exceptions

# Updated questions to satisfy the assertions in the next cell
questions = [
    "What is the main filling used in Masala Dosa?",
    "What spices are used to prepare Vegetable Biryani?",
    "How do you make Paneer Butter Masala?",
    "What ingredients are in the tempering for Lemon Rice?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(20)  # Stay within the free-tier rate limit.
    print("\n" + "=" * 60)

    # Wrap execution to gracefully handle 429 Rate Limits
    try:
        run_agent(q)
    except google.api_core.exceptions.ResourceExhausted:
        print("\nRate limit hit (429). Waiting 60 seconds to retry...")
        time.sleep(60)
        run_agent(q)


User Prompt: What is the main filling used in Masala Dosa?

Rate limit hit (429). Retrying in 30 seconds...
User Prompt: What is the main filling used in Masala Dosa?

Rate limit hit (429). Retrying in 60 seconds...
User Prompt: What is the main filling used in Masala Dosa?

Rate limit hit (429). Retrying in 120 seconds...
User Prompt: What is the main filling used in Masala Dosa?

Rate limit hit (429). Retrying in 180 seconds...
User Prompt: What is the main filling used in Masala Dosa?

Rate limit hit (429). Retrying in 180 seconds...
User Prompt: What is the main filling used in Masala Dosa?

Rate limit hit (429). Retrying in 180 seconds...
User Prompt: What is the main filling used in Masala Dosa?

Rate limit hit (429). Retrying in 180 seconds...
User Prompt: What is the main filling used in Masala Dosa?

Rate limit hit (429). Retrying in 180 seconds...
User Prompt: What is the main filling used in Masala Dosa?

Rate limit hit (429). Retrying in 180 seconds...
User Prompt: What is

In [ ]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)